# Qwen Model Optimization Pipeline for Google Colab

This notebook implements a full compression and optimization pipeline for Qwen LLMs, inspired by formal theorems in the Lean 4 theorem catalog:
- **CompressionPipeline**: 3-stage composition (quantize → prune → distill) with error bounds
- **QuantizationBounds**: Uniform quantization, Frobenius error bounds, adaptive step sizes
- **DistillationLoss**: Temperature-scaled KL divergence for knowledge distillation
- **CrystallizationTheory**: Weight clustering and sparse permutation optimization

## Workflow
1. Mount Google Drive for model caching
2. Download Qwen2.5-3B/7B from HuggingFace
3. Baseline benchmark (VRAM, tokens/sec, perplexity)
4. 4-bit quantization with bitsandbytes
5. GGUF conversion for llama.cpp inference
6. Optimized inference benchmark
7. Telemetry logging to Drive

**Recommended Runtime:** GPU (T4 or better)

In [ ]:
# ============================================================
# Section 1: Mount Google Drive for Model Caching
# ============================================================
from google.colab import drive
import os

DRIVE_PATH = "/content/drive/MyDrive/QwenCache"
os.makedirs(DRIVE_PATH, exist_ok=True)
drive.mount("/content/drive", force_remount=True)

print(f"Drive mounted. Cache directory: {DRIVE_PATH}")

In [ ]:
# ============================================================
# Section 2: Install Dependencies
# ============================================================
!pip install -q transformers accelerate bitsandbytes optimum[auto-gptq]
!pip install -q torch psutil tqdm huggingface_hub
!pip install -q datasets  # for perplexity benchmarking

# Optional: llama.cpp Python bindings for GGUF
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

print("Dependencies installed.")

In [ ]:
# ============================================================
# Section 3: Telemetry & Benchmarking Infrastructure
# ============================================================
import json, time, psutil, os, sys
from dataclasses import dataclass, asdict
from typing import Optional, List
from datetime import datetime
import torch

@dataclass
class TelemetryEntry:
    timestamp: str
    stage: str
    model_name: str
    quantization: str
    vram_mb: float
    tokens_per_sec_prefill: float
    tokens_per_sec_decode: float
    perplexity: Optional[float]
    latency_ttft_ms: float
    latency_tpot_ms: float
    notes: str = ""

class TelemetryLogger:
    def __init__(self, drive_path: str):
        self.drive_path = drive_path
        self.entries: List[TelemetryEntry] = []
        self.log_file = os.path.join(drive_path, "telemetry.json")
        self._load_existing()

    def _load_existing(self):
        if os.path.exists(self.log_file):
            with open(self.log_file, "r") as f:
                data = json.load(f)
                self.entries = [TelemetryEntry(**e) for e in data]
            print(f"Loaded {len(self.entries)} existing telemetry entries.")

    def log(self, entry: TelemetryEntry):
        self.entries.append(entry)
        with open(self.log_file, "w") as f:
            json.dump([asdict(e) for e in self.entries], f, indent=2)
        print(f"Logged: {entry.stage} | {entry.model_name} | {entry.quantization}")

    def summary(self):
        print("\n" + "="*60)
        print("Telemetry Summary")
        print("="*60)
        for e in self.entries:
            print(f"{e.timestamp} | {e.stage:20s} | {e.model_name:25s} | "
                  f"VRAM={e.vram_mb:7.1f}MB | Tok/s={e.tokens_per_sec_decode:6.2f}")

def get_vram_usage_mb() -> float:
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0.0

def get_gpu_name() -> str:
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return "CPU"

print(f"GPU: {get_gpu_name()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

logger = TelemetryLogger(DRIVE_PATH)

## Section 4: Model Download & Cache

Download Qwen2.5-3B-Instruct (or Qwen2.5-7B-Instruct) from HuggingFace and cache to Google Drive. The model is saved under `QwenCache/models/` so subsequent runs skip the download.

In [ ]:
# ============================================================
# Model Download with Drive Caching
# ============================================================
from huggingface_hub import snapshot_download
import os

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Switch to 7B if you have enough VRAM
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

MODEL_CACHE_DIR = os.path.join(DRIVE_PATH, "models", MODEL_NAME.replace("/", "_"))
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Cache directory: {MODEL_CACHE_DIR}")

if not os.path.exists(os.path.join(MODEL_CACHE_DIR, "config.json")):
    print("Downloading model (this may take a few minutes)...")
    snapshot_download(repo_id=MODEL_NAME, local_dir=MODEL_CACHE_DIR, local_dir_use_symlinks=False)
    print("Download complete.")
else:
    print("Model already cached in Drive. Skipping download.")

print(f"Cache size: {sum(os.path.getsize(os.path.join(dirpath, f)) for dirpath, _, files in os.walk(MODEL_CACHE_DIR) for f in files) / 1024**3:.2f} GB")

## Section 5: Baseline Benchmark (FP16)

Load the model in FP16 and measure baseline performance: VRAM usage, inference speed, and perplexity.

In [ ]:
# ============================================================
# Baseline Benchmark: FP16 Model
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time

torch.cuda.empty_cache()

print("Loading model in FP16 (baseline)...")
t0 = time.time()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_CACHE_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_CACHE_DIR, trust_remote_code=True)
load_time = time.time() - t0

print(f"Model loaded in {load_time:.1f}s")
print(f"VRAM after load: {get_vram_usage_mb():.1f} MB")

prompt = "Explain the Pythagorean theorem in one sentence:"
inputs = tokenizer(prompt, return_tensors="pt").to(model_fp16.device)

with torch.no_grad():
    _ = model_fp16.generate(**inputs, max_new_tokens=10, do_sample=False)
torch.cuda.synchronize()

t0 = time.time()
with torch.no_grad():
    output = model_fp16.generate(**inputs, max_new_tokens=1, do_sample=False)
ttft = (time.time() - t0) * 1000

gen_tokens = 50
t0 = time.time()
with torch.no_grad():
    output = model_fp16.generate(**inputs, max_new_tokens=gen_tokens, do_sample=False, use_cache=True)
total_gen_time = time.time() - t0

output_text = tokenizer.decode(output[0], skip_special_tokens=True)
tpot = (total_gen_time * 1000) / gen_tokens
prefill_tok_s = inputs.input_ids.shape[1] / (ttft / 1000)
decode_tok_s = gen_tokens / total_gen_time

print(f"\n--- Baseline FP16 Results ---")
print(f"VRAM: {get_vram_usage_mb():.1f} MB")
print(f"Prefill speed: {prefill_tok_s:.1f} tok/s")
print(f"Decode speed:  {decode_tok_s:.1f} tok/s")
print(f"TTFT: {ttft:.1f} ms")
print(f"TPOT: {tpot:.1f} ms")
print(f"Output: {output_text[:120]}...")

logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage="baseline_fp16",
    model_name=MODEL_NAME,
    quantization="fp16",
    vram_mb=get_vram_usage_mb(),
    tokens_per_sec_prefill=prefill_tok_s,
    tokens_per_sec_decode=decode_tok_s,
    perplexity=None,
    latency_ttft_ms=ttft,
    latency_tpot_ms=tpot,
    notes=f"Load time: {load_time:.1f}s"
))

In [ ]:
# ============================================================
# Perplexity Benchmark (Wikitext-2 sample)
# ============================================================
from datasets import load_dataset
import torch
import torch.nn.functional as F

try:
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
    text_samples = [t for t in ds["text"] if len(t) > 100][:20]
    encodings = tokenizer("\n\n".join(text_samples), return_tensors="pt")
    max_length = model_fp16.config.max_position_embeddings if hasattr(model_fp16.config, "max_position_embeddings") else 2048
    stride = 512
    seq_len = encodings.input_ids.size(1)
    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(model_fp16.device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        with torch.no_grad():
            outputs = model_fp16(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss * trg_len
        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break
    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    print(f"Perplexity (FP16): {ppl.item():.2f}")
    logger.entries[-1].perplexity = ppl.item()
    logger._save()
except Exception as e:
    print(f"Perplexity benchmark skipped: {e}")
    ppl = None

## Section 6: 4-bit Quantization (NF4)

Apply 4-bit Normal Float quantization using `bitsandbytes`. This is the first stage of the compression pipeline, directly inspired by the `QuantizationBounds.lean` theorem on uniform quantization error bounds.

For a model with weight range $W$, 4-bit quantization gives step size $\delta = W / 16$, and the per-weight error is bounded by $|w - Q(w)| \leq \delta/2$.

In [ ]:
# ============================================================
# 4-bit Quantization with bitsandbytes (NF4)
# ============================================================
from transformers import BitsAndBytesConfig

torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading model in 4-bit NF4...")
t0 = time.time()
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_CACHE_DIR,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
load_time_4bit = time.time() - t0

print(f"4-bit model loaded in {load_time_4bit:.1f}s")
print(f"VRAM after 4-bit load: {get_vram_usage_mb():.1f} MB")

inputs = tokenizer(prompt, return_tensors="pt").to(model_4bit.device)
with torch.no_grad():
    _ = model_4bit.generate(**inputs, max_new_tokens=10, do_sample=False)
torch.cuda.synchronize()

t0 = time.time()
with torch.no_grad():
    output = model_4bit.generate(**inputs, max_new_tokens=1, do_sample=False)
ttft_4 = (time.time() - t0) * 1000

t0 = time.time()
with torch.no_grad():
    out_4 = model_4bit.generate(**inputs, max_new_tokens=gen_tokens, do_sample=False, use_cache=True)
total_4 = time.time() - t0
out_4_text = tokenizer.decode(out_4[0], skip_special_tokens=True)

prefill_4 = inputs.input_ids.shape[1] / (ttft_4 / 1000)
decode_4 = gen_tokens / total_4

print(f"\n--- 4-bit NF4 Results ---")
print(f"VRAM: {get_vram_usage_mb():.1f} MB")
print(f"Prefill speed: {prefill_4:.1f} tok/s")
print(f"Decode speed:  {decode_4:.1f} tok/s")
print(f"TTFT: {ttft_4:.1f} ms")
print(f"TPOT: {(total_4*1000)/gen_tokens:.1f} ms")
print(f"VRAM reduction: {(1 - get_vram_usage_mb()/logger.entries[0].vram_mb)*100:.1f}%")
print(f"Output: {out_4_text[:120]}...")

logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage="quantize_4bit_nf4",
    model_name=MODEL_NAME,
    quantization="nf4",
    vram_mb=get_vram_usage_mb(),
    tokens_per_sec_prefill=prefill_4,
    tokens_per_sec_decode=decode_4,
    perplexity=None,
    latency_ttft_ms=ttft_4,
    latency_tpot_ms=(total_4*1000)/gen_tokens,
    notes=f"Load time: {load_time_4bit:.1f}s | Double quant enabled"
))

## Section 7: GGUF Conversion for llama.cpp

Convert the model to GGUF format for highly optimized CPU/GPU inference. This enables aggressive quantization (Q4_K_M, Q3_K_M) and runs efficiently with `llama-cpp-python`.

We use the `convert_hf_to_gguf.py` script from the `llama.cpp` repo.

In [ ]:
# ============================================================
# GGUF Conversion
# ============================================================
import subprocess, os

GGUF_DIR = os.path.join(DRIVE_PATH, "gguf")
os.makedirs(GGUF_DIR, exist_ok=True)

LLAMA_CPP_DIR = os.path.join(DRIVE_PATH, "llama.cpp")
if not os.path.exists(LLAMA_CPP_DIR):
    print("Cloning llama.cpp...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", LLAMA_CPP_DIR], check=True)
else:
    print("llama.cpp already cloned.")

!pip install -q -r {LLAMA_CPP_DIR}/requirements.txt

GGUF_OUT = os.path.join(GGUF_DIR, f"{MODEL_NAME.replace('/', '_')}.gguf")

print("Converting to GGUF (this may take a few minutes)...")
result = subprocess.run(
    ["python3", os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"), MODEL_CACHE_DIR, "--outfile", GGUF_OUT, "--outtype", "f16"],
    capture_output=True, text=True
)
print(result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
else:
    size_gb = os.path.getsize(GGUF_OUT) / 1024**3
    print(f"GGUF created: {GGUF_OUT} ({size_gb:.2f} GB)")

In [ ]:
# ============================================================
# Quantize GGUF to Q4_K_M and Q3_K_M
# ============================================================
QUANT_LEVELS = ["Q4_K_M", "Q3_K_M"]

quant_results = {}

for quant in QUANT_LEVELS:
    out_path = GGUF_OUT.replace(".gguf", f"_{quant}.gguf")
    if os.path.exists(out_path):
        print(f"{quant} already exists, skipping.")
    else:
        print(f"Quantizing to {quant}...")
        result = subprocess.run(
            [os.path.join(LLAMA_CPP_DIR, "llama-quantize"), GGUF_OUT, out_path, quant],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"llama-quantize not found, building...")
            build_res = subprocess.run(["make", "-C", LLAMA_CPP_DIR, "-j"], capture_output=True, text=True)
            result = subprocess.run(
                [os.path.join(LLAMA_CPP_DIR, "llama-quantize"), GGUF_OUT, out_path, quant],
                capture_output=True, text=True
            )
    if os.path.exists(out_path):
        size_gb = os.path.getsize(out_path) / 1024**3
        quant_results[quant] = {"path": out_path, "size_gb": size_gb}
        print(f"  {quant}: {size_gb:.2f} GB")

print("\nQuantization complete.")

## Section 8: GGUF Inference Benchmark

Load the quantized GGUF models with `llama-cpp-python` and benchmark inference speed and VRAM usage.

In [ ]:
# ============================================================
# GGUF Inference with llama-cpp-python
# ============================================================
from llama_cpp import Llama
import time

for quant_name, info in quant_results.items():
    print(f"\n{'='*60}")
    print(f"Benchmarking {quant_name} ({info['size_gb']:.2f} GB)")
    print("="*60)

    t0 = time.time()
    llm = Llama(
        model_path=info["path"],
        n_ctx=2048,
        n_gpu_layers=-1,
        verbose=False,
    )
    load_t = time.time() - t0

    _ = llm(prompt, max_tokens=10, temperature=0)

    gen_tokens = 50
    t0 = time.time()
    out = llm(prompt, max_tokens=gen_tokens, temperature=0)
    total_t = time.time() - t0

    output_text = out["choices"][0]["text"]
    tok_s = gen_tokens / total_t

    print(f"Load time: {load_t:.1f}s")
    print(f"Decode speed: {tok_s:.1f} tok/s")
    print(f"Total time: {total_t:.1f}s")
    print(f"Output: {output_text[:120]}...")

    logger.log(TelemetryEntry(
        timestamp=datetime.utcnow().isoformat(),
        stage="gguf_inference",
        model_name=MODEL_NAME,
        quantization=quant_name,
        vram_mb=get_vram_usage_mb(),
        tokens_per_sec_prefill=0,
        tokens_per_sec_decode=tok_s,
        perplexity=None,
        latency_ttft_ms=0,
        latency_tpot_ms=(total_t * 1000) / gen_tokens,
        notes=f"GGUF size: {info['size_gb']:.2f}GB | Load: {load_t:.1f}s"
    ))

## Section 9: Telemetry Summary & Comparison

Compare all optimization stages side-by-side: VRAM usage, inference speed, and compression ratio.

In [ ]:
# ============================================================
# Summary Table & Visualization
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

logger.summary()

data = []
for e in logger.entries:
    data.append({
        "Stage": e.stage,
        "Quantization": e.quantization,
        "VRAM (MB)": e.vram_mb,
        "Decode Tok/s": e.tokens_per_sec_decode,
        "TPOT (ms)": e.latency_tpot_ms,
        "Perplexity": e.perplexity,
    })

df = pd.DataFrame(data)
print("\n" + "="*80)
print(df.to_string(index=False))
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(df["Stage"], df["VRAM (MB)"], color="steelblue")
axes[0].set_ylabel("VRAM (MB)")
axes[0].set_title("VRAM Usage by Stage")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(df["Stage"], df["Decode Tok/s"], color="forestgreen")
axes[1].set_ylabel("Tokens/sec")
axes[1].set_title("Decode Throughput")
axes[1].tick_params(axis="x", rotation=45)

axes[2].bar(df["Stage"], df["TPOT (ms)"], color="coral")
axes[2].set_ylabel("ms/token")
axes[2].set_title("Time Per Output Token")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_PATH, "benchmark_chart.png"), dpi=150)
plt.show()

print(f"\nChart saved to: {os.path.join(DRIVE_PATH, 'benchmark_chart.png')}")
print(f"Telemetry saved to: {logger.log_file}")

## Section 10: Flash Attention Optimization

Enable Flash Attention 2 in the transformers pipeline for faster, memory-efficient attention. This implements the sub-quadratic attention mechanisms theorized in `SubQuadraticAttention.lean`.

In [ ]:
# ============================================================
# Flash Attention 2 Benchmark
# ============================================================
try:
    !pip install -q flash-attn --no-build-isolation
    print("Flash Attention installed.")
except Exception as e:
    print(f"Flash Attention install skipped: {e}")

torch.cuda.empty_cache()

print("Loading model with Flash Attention 2 (if available)...")
try:
    model_fa = AutoModelForCausalLM.from_pretrained(
        MODEL_CACHE_DIR,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="flash_attention_2",
    )
    print("Flash Attention 2 active.")
except Exception as e:
    print(f"Flash Attention 2 not available, using eager: {e}")
    model_fa = AutoModelForCausalLM.from_pretrained(
        MODEL_CACHE_DIR,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

inputs = tokenizer(prompt, return_tensors="pt").to(model_fa.device)
with torch.no_grad():
    _ = model_fa.generate(**inputs, max_new_tokens=10, do_sample=False)
torch.cuda.synchronize()

t0 = time.time()
with torch.no_grad():
    out_fa = model_fa.generate(**inputs, max_new_tokens=gen_tokens, do_sample=False, use_cache=True)
total_fa = time.time() - t0

tok_s_fa = gen_tokens / total_fa
print(f"\n--- Flash Attention Results ---")
print(f"VRAM: {get_vram_usage_mb():.1f} MB")
print(f"Decode speed: {tok_s_fa:.1f} tok/s")
print(f"Improvement over baseline: {(tok_s_fa / logger.entries[0].tokens_per_sec_decode - 1)*100:.1f}%")

logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage="flash_attention",
    model_name=MODEL_NAME,
    quantization="bf16+fa2",
    vram_mb=get_vram_usage_mb(),
    tokens_per_sec_prefill=0,
    tokens_per_sec_decode=tok_s_fa,
    perplexity=None,
    latency_ttft_ms=0,
    latency_tpot_ms=(total_fa*1000)/gen_tokens,
    notes="Flash Attention 2 enabled"
))

## Section 11: Knowledge Distillation (Stage 3 of Compression Pipeline)

Distill the teacher model (Qwen2.5-3B/7B) into a smaller student (Qwen2.5-0.5B) using temperature-scaled KL divergence, as formalized in `DistillationLoss.lean`.

The distillation loss is:
$$\mathcal{L} = (1-\alpha) \cdot \mathcal{L}_{\text{CE}} + \alpha \cdot T^2 \cdot \mathcal{L}_{\text{KL}}$$

where $T$ is the temperature and $\alpha$ balances hard and soft labels.

In [ ]:
# ============================================================
# Knowledge Distillation Setup
# ============================================================
TEACHER_NAME = MODEL_NAME

print("Loading teacher for distillation demo...")

# Reuse the baseline model as teacher (already in memory from Cell 7)
if 'model_fp16' in globals():
    teacher = model_fp16
    print("Reusing baseline model from memory as teacher.")
else:
    teacher = AutoModelForCausalLM.from_pretrained(
        MODEL_CACHE_DIR,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

# Avoid torchvision/torch version conflicts by NOT loading a 0.5B HF student.
# Instead, create a small tropical student directly.
from qwen_optimizer.tropical import TropicalModel

student_config = {
    "vocab_size": teacher.config.vocab_size,
    "d_model": 512,
    "num_layers": 6,
    "num_heads": 8,
    "d_ff": 1024,
    "max_seq_len": 2048,
    "dropout": 0.1,
    "hard_attention": False,
}
student = TropicalModel(**student_config)
student = student.to(device)
student.eval()

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print(f"Teacher params: {teacher_params/1e6:.1f}M")
print(f"Student params: {student_params/1e6:.1f}M")
print(f"Compression ratio: {teacher_params / student_params:.1f}x")

# Quick sanity: run student forward pass
with torch.no_grad():
    test_ids = torch.randint(0, tokenizer.vocab_size, (1, 16), device=device)
    logits = student(test_ids, is_causal=True)
    print(f"Student forward pass OK: {logits.shape}")

T = 2.0
alpha = 0.5
print(f"\nDistillation config: T={T}, alpha={alpha}")
print("Synthetic dataset generation + distillation loop would go here.")
print("See qwen_optimizer/distill.py for a full implementation.")

# ============================================================
# Section 13: Tropical Compression Architecture
# ============================================================
from qwen_optimizer.tropical import (
    TropicalModel,
    TropicalLinear,
    TropicalAttention,
    TropicalFFN,
    tropical_matmul,
    tropical_dot_product,
    crystallization_penalty,
    sheffer_nand,
    tropical_to_sheffer,
)

# Create a tropical student (same config as distillation student)
tropical_config = {
    "vocab_size": teacher.config.vocab_size,
    "d_model": 512,
    "num_layers": 6,
    "num_heads": 8,
    "d_ff": 1024,
    "max_seq_len": 2048,
    "dropout": 0.0,
    "hard_attention": False,
}
tropical_model = TropicalModel(**tropical_config)
tropical_model = tropical_model.to(device)
tropical_model.eval()

print(f"Tropical model params: {sum(p.numel() for p in tropical_model.parameters())/1e6:.1f}M")
print(f"Tropical operations use min-plus algebra (no FP multiplications in linear layers)")

# Forward pass benchmark
with torch.no_grad():
    test_ids = torch.randint(0, teacher.config.vocab_size, (1, 16), device=device)
    t0 = time.time()
    for _ in range(10):
        logits = tropical_model(test_ids, is_causal=True)
    torch.cuda.synchronize()
    avg_t = (time.time() - t0) / 10 * 1000

print(f"Tropical forward pass (16 tokens): {avg_t:.2f} ms")
print(f"Tropical output shape: {logits.shape}")

# ============================================================
# Section 14: Tropical Distillation Training Loop
# ============================================================
# This cell runs a short distillation loop from the teacher (Qwen2.5-3B)
# into the tropical student created above.

from qwen_optimizer.tropical_train import generate_synthetic_data, TextDataset, train_tropical_model
from qwen_optimizer.tropical import TropicalModel

# Ensure teacher is available
if 'teacher' not in globals():
    teacher = model_fp16

# Create fresh tropical student
tropical_student = TropicalModel(
    vocab_size=teacher.config.vocab_size,
    d_model=512,
    num_layers=6,
    num_heads=8,
    d_ff=1024,
    max_seq_len=2048,
    dropout=0.1,
    hard_attention=False,
)
tropical_student = tropical_student.to(device)

# Generate a small synthetic dataset (adjust num_samples for longer runs)
print("Generating synthetic dataset...")
synthetic_texts = generate_synthetic_data(
    teacher,
    tokenizer,
    num_samples=20,
    max_length=128,
)
train_dataset = TextDataset(synthetic_texts, tokenizer, max_length=128)
print(f"Dataset size: {len(train_dataset)} samples")

# Run a mini training loop (1 epoch for Colab demo)
print("\nStarting tropical distillation training...")
trained_model = train_tropical_model(
    teacher=teacher,
    tropical_model=tropical_student,
    tokenizer=tokenizer,
    dataset=train_dataset,
    epochs=1,
    batch_size=2,
    lr=5e-5,
    temperature=2.0,
    alpha=0.5,
    crystallization_weight=0.01,
    device=device,
    output_dir=os.path.join(DRIVE_PATH, "tropical_model"),
)

# Benchmark the trained model
bench = BenchmarkSuite(tokenizer, device=device)
train_result = bench.run_inference_benchmark(
    trained_model,
    stage_name="tropical_distilled",
    quantization_label="tropical_fp16",
    notes="1_epoch",
)

logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage=train_result.stage,
    model_name="TropicalStudent",
    quantization=train_result.quantization,
    vram_mb=train_result.vram_mb,
    tokens_per_sec_prefill=train_result.tokens_per_sec_prefill,
    tokens_per_sec_decode=train_result.tokens_per_sec_decode,
    perplexity=None,
    latency_ttft_ms=train_result.latency_ttft_ms,
    latency_tpot_ms=train_result.latency_tpot_ms,
    notes=train_result.notes,
))

print(f"\nTrained tropical model saved to: {os.path.join(DRIVE_PATH, 'tropical_model')}")


In [ ]:
# ============================================================
# Section 15: Crystallization & Sheffer Logic Mapping
# ============================================================
from qwen_optimizer.tropical import crystallization_penalty, sheffer_nand, tropical_to_sheffer

# Use the trained model from the previous cell, or create a fresh one
if 'trained_model' not in globals():
    crystallized = TropicalModel(
        vocab_size=teacher.config.vocab_size,
        d_model=512,
        num_layers=6,
        num_heads=8,
        d_ff=1024,
        max_seq_len=2048,
        dropout=0.0,
        hard_attention=True,
    )
    crystallized = crystallized.to(device)
else:
    crystallized = trained_model
    crystallized.eval()

# Compute crystallization penalty before
pre_penalty = sum(crystallization_penalty(p) for p in crystallized.parameters())
print(f"Crystallization penalty BEFORE: {pre_penalty.item():.4f}")

# Crystallize all weights to {-1, 0, 1}
print("Crystallizing weights...")
crystallized.crystallize()

# Compute penalty after (should be ~0)
post_penalty = sum(crystallization_penalty(p) for p in crystallized.parameters())
print(f"Crystallization penalty AFTER:  {post_penalty.item():.4f}")

# Count discrete values
total_params = sum(p.numel() for p in crystallized.parameters())
neg1 = sum((p == -1).sum().item() for p in crystallized.parameters())
zeros = sum((p == 0).sum().item() for p in crystallized.parameters())
pos1 = sum((p == 1).sum().item() for p in crystallized.parameters())
print(f"Weight distribution: {-1}: {neg1/total_params*100:.2f}%, 0: {zeros/total_params*100:.2f}%, +1: {pos1/total_params*100:.2f}%")

# Benchmark crystallized model
bench = BenchmarkSuite(tokenizer, device=device)
cryst_result = bench.run_inference_benchmark(
    crystallized,
    stage_name="crystallized",
    quantization_label="ternary",
    notes="hard_attention=True",
)

logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage=cryst_result.stage,
    model_name="CrystallizedStudent",
    quantization=cryst_result.quantization,
    vram_mb=cryst_result.vram_mb,
    tokens_per_sec_prefill=cryst_result.tokens_per_sec_prefill,
    tokens_per_sec_decode=cryst_result.tokens_per_sec_decode,
    perplexity=None,
    latency_ttft_ms=cryst_result.latency_ttft_ms,
    latency_tpot_ms=cryst_result.latency_tpot_ms,
    notes=cryst_result.notes,
))

# Sheffer NAND demonstration
print("\n--- Sheffer NAND Logic Demo ---")
a = torch.tensor([0.0, 0.0, 1.0, 1.0], device=device)
b = torch.tensor([0.0, 1.0, 0.0, 1.0], device=device)
nand_out = sheffer_nand(a, b)
print(f"a     = {a.tolist()}")
print(f"b     = {b.tolist()}")
print(f"NAND  = {nand_out.tolist()}")
print("(NAND is functionally complete — any boolean circuit can be built from it)")

# Save crystallized model
import torch
cryst_path = os.path.join(DRIVE_PATH, "crystallized_model.pt")
torch.save(crystallized.state_dict(), cryst_path)
print(f"\nCrystallized model saved to: {cryst_path}")

logger.summary()
print(f"\nTelemetry saved to: {logger.log_file}")
print("Done!")

## Section 16: Qwen3.5-35B-A3B MoE Optimization

The 35B-A3B model is a Sparse Mixture-of-Experts (MoE) with 35B total / 3B active parameters. At 4-bit it needs ~17.5GB weights + KV cache, so T4 (16GB) requires aggressive optimization.

MoE-specific techniques:
- **Mixed quantization**: attention layers at Q8_0, expert FFNs at IQ4_K
- **Expert offloading**: route unused experts to CPU RAM
- **Dynamic batching**: batch multiple tokens per expert call
- **YaRN scaling**: extend 262K context without recomputing RoPE


In [ ]:
# ============================================================
# MoE Model Download & Mixed Quantization Recipe
# ============================================================
MOE_MODEL = "Qwen/Qwen3.5-35B-A3B"
MOE_CACHE = os.path.join(DRIVE_PATH, "models", MOE_MODEL.replace("/", "_"))
os.makedirs(MOE_CACHE, exist_ok=True)

if not os.path.exists(os.path.join(MOE_CACHE, "config.json")):
    print(f"Downloading {MOE_MODEL} (this will take ~10-15 min)...")
    snapshot_download(repo_id=MOE_MODEL, local_dir=MOE_CACHE, local_dir_use_symlinks=False)
else:
    print("MoE model already cached.")

# Mixed quantization: Q8_0 for attention, IQ4_K for experts
print("\nRecommended GGUF quant recipe for 35B-A3B:")
print("  - Attention proj: Q8_0 (preserves accuracy)")
print("  - Expert FFN:   IQ4_K (aggressive compression)")
print("  - Gate router:  Q6_K (routing must be precise)")
print("  - KV cache:     Q4_0 (save VRAM during long context)")
print("\nUse llama.cpp convert script with --outtype iq4_xs for experts.")


In [ ]:
# ============================================================
# MoE Inference with llama-cpp-python + Offloading
# ============================================================
try:
    from llama_cpp import Llama

    moe_gguf = os.path.join(GGUF_DIR, "Qwen_Qwen3.5-35B-A3B_Q4_K_M.gguf")

    if os.path.exists(moe_gguf):
        print(f"Loading MoE GGUF: {moe_gguf}")
        llm_moe = Llama(
            model_path=moe_gguf,
            n_ctx=8192,
            n_gpu_layers=-1,
            offload_kqv=True,
            split_mode=1,
            verbose=False,
        )

        t0 = time.time()
        out_moe = llm_moe(prompt, max_tokens=30, temperature=0)
        moe_time = time.time() - t0
        text = out_moe["choices"][0]["text"][:100]
        print(f"MoE output ({moe_time:.1f}s): {text}...")

        logger.log(TelemetryEntry(
            timestamp=datetime.utcnow().isoformat(),
            stage="moe_inference",
            model_name=MOE_MODEL,
            quantization="Q4_K_M",
            vram_mb=get_vram_usage_mb(),
            tokens_per_sec_prefill=0,
            tokens_per_sec_decode=30 / moe_time,
            perplexity=None,
            latency_ttft_ms=0,
            latency_tpot_ms=(moe_time * 1000) / 30,
            notes="MoE 35B/3B active",
        ))
    else:
        print(f"MoE GGUF not found at {moe_gguf}")
        print("Convert the model first using the GGUF cells above.")

except ImportError:
    print("llama-cpp-python not available. Skipping MoE inference.")


In [ ]:
# ============================================================
# Expert Offloading Simulation
# ============================================================
def simulate_expert_offloading(
    num_experts: int = 256,
    active_experts: int = 8,
    vram_per_expert_mb: float = 60.0,
) -> dict:
    total_expert_vram = num_experts * vram_per_expert_mb
    active_vram = active_experts * vram_per_expert_mb
    offloaded = total_expert_vram - active_vram
    return {
        "total_experts": num_experts,
        "active_experts": active_experts,
        "total_expert_vram_mb": total_expert_vram,
        "active_vram_mb": active_vram,
        "offloaded_vram_mb": offloaded,
        "offload_ratio": offloaded / total_expert_vram,
    }

moe_sim = simulate_expert_offloading()
print("MoE Expert Offloading Simulation:")
for k, v in moe_sim.items():
    print(f"  {k}: {v:.2f}")

print("\nFor Qwen3.5-35B-A3B:")
print("  - 256 experts, 8 active per token")
print("  - Offloading 248 experts saves ~15GB of VRAM")
print("  - Only 8 experts (~480MB) stay on GPU")


## Section 17: Batched Throughput with vLLM / SGLang

For serving scenarios, batching multiple requests dramatically improves throughput.
vLLM and SGLang use PagedAttention to batch efficiently without padding waste.

This section installs vLLM and runs a batched benchmark.


In [ ]:
# ============================================================
# Install vLLM
# ============================================================
try:
    import vllm
    print("vLLM already installed.")
except ImportError:
    print("Installing vLLM (this may take 3-5 minutes)...")
    !pip install -q vllm

try:
    import sglang
    print("SGLang already installed.")
except ImportError:
    print("SGLang not installed (optional).")


In [ ]:
# ============================================================
# vLLM Batched Benchmark
# ============================================================
try:
    from vllm import LLM, SamplingParams

    prompts = [
        "Explain quantum computing in one sentence:",
        "The Pythagorean theorem states",
        "In machine learning, a neural network",
        "The capital of Japan is",
        "To solve for x: 3x + 5 = 20",
    ] * 4  # 20 prompts for batching

    print(f"Loading vLLM with {MODEL_NAME}...")
    llm_vllm = LLM(
        model=MODEL_CACHE_DIR,
        tensor_parallel_size=1,
        gpu_memory_utilization=0.9,
        max_model_len=2048,
    )

    sampling_params = SamplingParams(
        temperature=0,
        max_tokens=50,
    )

    t0 = time.time()
    outputs = llm_vllm.generate(prompts, sampling_params)
    total_time = time.time() - t0

    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_tokens / total_time

    print(f"\n--- vLLM Batched Results ---")
    print(f"Batch size: {len(prompts)}")
    print(f"Total tokens generated: {total_tokens}")
    print(f"Total time: {total_time:.1f}s")
    print(f"Throughput: {throughput:.1f} tok/s")
    print(f"VRAM: {get_vram_usage_mb():.1f} MB")

    logger.log(TelemetryEntry(
        timestamp=datetime.utcnow().isoformat(),
        stage="vllm_batched",
        model_name=MODEL_NAME,
        quantization="fp16",
        vram_mb=get_vram_usage_mb(),
        tokens_per_sec_prefill=0,
        tokens_per_sec_decode=throughput,
        perplexity=None,
        latency_ttft_ms=0,
        latency_tpot_ms=(total_time * 1000) / total_tokens,
        notes=f"batch={len(prompts)} | PagedAttention",
    ))

except ImportError:
    print("vLLM not available. Skipping batched benchmark.")


## Conclusion

This notebook demonstrated a full optimization pipeline for Qwen LLMs:

| Stage | VRAM Reduction | Speed Impact | Quality |
|-------|---------------|-------------|---------|
| FP16 Baseline | — | Baseline | Best |
| 4-bit NF4 | ~60-70% | Minimal | Excellent |
| GGUF Q4_K_M | ~75% | Fast | Very Good |
| GGUF Q3_K_M | ~85% | Fast | Good |
| Flash Attention | Slight | +20-40% | Same |
| Tropical Student | Model-dependent | No FP mults | Depends on distillation |
| Crystallized | Model-dependent | Ternary ops | Depends on crystallization |

All telemetry is saved to your Google Drive at:
- 
- 

## Next Steps
1. Run the full perplexity benchmark on a larger dataset
2. Extend distillation to more epochs for better quality
3. Add pruning (structured + unstructured) as Stage 2
4. Extend to Qwen3.5-35B-A3B on Colab A100
5. Use  or  for batched throughput benchmarks

## References to Lean Theorems
-  — multi-stage error composition
-  — quantization error bound
-  — temperature-scaled distillation loss
-  — weight crystallization
-  — tropical semiring primitives
-  — L1 distance attention
-  — Sheffer stroke logic mapping


## Section 18: Full Perplexity Benchmark\n\nRun a comprehensive perplexity evaluation across all model variants using Wikitext-2 validation.\nThis measures quality degradation from quantization, distillation, and crystallization.

In [ ]:
# ============================================================\n# Full Perplexity Benchmark (Wikitext-2)\n# ============================================================\nfrom datasets import load_dataset\nimport torch\nimport torch.nn.functional as F\nimport gc\n\ndef compute_perplexity(model, tokenizer, device='cuda', max_samples=50, stride=512):\n    \"\"\"Compute perplexity on Wikitext-2 validation subset.\"\"\"\n    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='validation')\n    texts = [t for t in ds['text'] if len(t) > 50][:max_samples]\n    enc = tokenizer('\\n\\n'.join(texts), return_tensors='pt')\n    seq_len = enc.input_ids.size(1)\n    max_len = getattr(model.config, 'max_position_embeddings', 2048) if hasattr(model, 'config') else 2048\n\n    nlls = []\n    prev_end = 0\n    model.eval()\n    for begin in range(0, seq_len, stride):\n        end = min(begin + max_len, seq_len)\n        trg = end - prev_end\n        ids = enc.input_ids[:, begin:end].to(device)\n        target = ids.clone()\n        target[:, :-trg] = -100\n        with torch.no_grad():\n            out = model(ids, labels=target) if hasattr(model, 'config') else model(ids, causal=True)\n            if isinstance(out, tuple):\n                loss = F.cross_entropy(out[0].view(-1, out[0].size(-1)), target.view(-1), ignore_index=-100)\n            elif hasattr(out, 'loss'):\n                loss = out.loss\n            else:\n                loss = F.cross_entropy(out.view(-1, out.size(-1)), target.view(-1), ignore_index=-100)\n        nlls.append(loss * trg)\n        prev_end = end\n        if end == seq_len:\n            break\n    ppl = torch.exp(torch.stack(nlls).sum() / end)\n    return ppl.item()\n\n# --- Baseline FP16 ---\nif 'model_fp16' in globals():\n    print('Computing baseline FP16 perplexity...')\n    ppl_base = compute_perplexity(model_fp16, tokenizer, device=model_fp16.device, max_samples=50)\n    print(f'Baseline FP16 PPL: {ppl_base:.2f}')\n    logger.entries[0].perplexity = ppl_base\n    logger._save()\nelse:\n    print('Baseline model not in memory. Skipping baseline PPL.')\n    ppl_base = None\n\n# --- 4-bit NF4 ---\nif 'model_4bit' in globals():\n    print('Computing 4-bit NF4 perplexity...')\n    ppl_4bit = compute_perplexity(model_4bit, tokenizer, device=model_4bit.device, max_samples=50)\n    print(f'4-bit NF4 PPL: {ppl_4bit:.2f}')\n    # Find the quantize entry and update it\n    for e in logger.entries:\n        if e.stage == 'quantize_4bit_nf4':\n            e.perplexity = ppl_4bit\n    logger._save()\nelse:\n    print('4-bit model not in memory. Skipping 4-bit PPL.')\n    ppl_4bit = None\n\n# --- Tropical Student ---\nif 'trained_model' in globals():\n    print('Computing tropical student perplexity...')\n    ppl_tropical = compute_perplexity(trained_model, tokenizer, device=device, max_samples=20)\n    print(f'Tropical Student PPL: {ppl_tropical:.2f}')\n    logger.log(TelemetryEntry(\n        timestamp=datetime.utcnow().isoformat(),\n        stage='perplexity_tropical',\n        model_name='TropicalStudent',\n        quantization='tropical_fp16',\n        vram_mb=get_vram_usage_mb(),\n        tokens_per_sec_prefill=0,\n        tokens_per_sec_decode=0,\n        perplexity=ppl_tropical,\n        latency_ttft_ms=0,\n        latency_tpot_ms=0,\n        notes='Wikitext-2 validation',\n    ))\nelse:\n    print('Trained tropical model not available. Run distillation first.')\n    ppl_tropical = None\n\n# --- Crystallized ---\nif 'crystallized' in globals():\n    print('Computing crystallized model perplexity...')\n    ppl_cryst = compute_perplexity(crystallized, tokenizer, device=device, max_samples=20)\n    print(f'Crystallized PPL: {ppl_cryst:.2f}')\n    logger.log(TelemetryEntry(\n        timestamp=datetime.utcnow().isoformat(),\n        stage='perplexity_crystallized',\n        model_name='CrystallizedStudent',\n        quantization='ternary',\n        vram_mb=get_vram_usage_mb(),\n        tokens_per_sec_prefill=0,\n        tokens_per_sec_decode=0,\n        perplexity=ppl_cryst,\n        latency_ttft_ms=0,\n        latency_tpot_ms=0,\n        notes='Wikitext-2 validation',\n    ))\nelse:\n    print('Crystallized model not available.')\n    ppl_cryst = None\n\n# Summary table\nprint('\\n' + '='*60)\nprint('Perplexity Summary (lower is better)')\nprint('='*60)\nif ppl_base:\n    print(f'  Baseline FP16:  {ppl_base:.2f}')\nif ppl_4bit:\n    print(f'  4-bit NF4:      {ppl_4bit:.2f}  (delta: {ppl_4bit-ppl_base:+.2f})')\nif ppl_tropical:\n    print(f'  Tropical:       {ppl_tropical:.2f}  (delta: {ppl_tropical-ppl_base:+.2f})')\nif ppl_cryst:\n    print(f'  Crystallized:   {ppl_cryst:.2f}  (delta: {ppl_cryst-ppl_base:+.2f})')\nprint('='*60)\n

## Section 19: GGUF CPU Inference\n\nLoad quantized GGUF models on CPU for ultra-low-latency inference without GPU.\n`llama-cpp-python` with `n_gpu_layers=0` forces pure CPU execution.

In [ ]:
# ============================================================\n# GGUF CPU Inference Benchmark\n# ============================================================\ntry:\n    from llama_cpp import Llama\n\n    # Use Q4_K_M if available, otherwise fallback to Q3_K_M\n    cpu_gguf = None\n    for q in ['Q4_K_M', 'Q3_K_M']:\n        path = GGUF_OUT.replace('.gguf', f'_{q}.gguf')\n        if os.path.exists(path):\n            cpu_gguf = path\n            print(f'Using {q} for CPU inference')\n            break\n    if cpu_gguf is None:\n        print('No GGUF quantized model found. Run Section 7 first.')\n    else:\n        print(f'Loading GGUF on CPU: {cpu_gguf}')\n        t0 = time.time()\n        llm_cpu = Llama(\n            model_path=cpu_gguf,\n            n_ctx=2048,\n            n_gpu_layers=0,  # Force CPU\n            n_threads=4,     # Adjust based on CPU cores\n            verbose=False,\n        )\n        load_t = time.time() - t0\n        print(f'CPU model loaded in {load_t:.1f}s')\n\n        # Warmup\n        _ = llm_cpu(prompt, max_tokens=10, temperature=0)\n\n        # Benchmark\n        gen_tokens = 50\n        t0 = time.time()\n        out_cpu = llm_cpu(prompt, max_tokens=gen_tokens, temperature=0)\n        total_t = time.time() - t0\n        tok_s = gen_tokens / total_t\n\n        text = out_cpu['choices'][0]['text']\n        print(f'\\n--- CPU Inference Results ---')\n        print(f'Load time: {load_t:.1f}s')\n        print(f'Decode speed: {tok_s:.1f} tok/s')\n        print(f'TPOT: {(total_t*1000)/gen_tokens:.1f} ms')\n        print(f'Output: {text[:120]}...')\n\n        logger.log(TelemetryEntry(\n            timestamp=datetime.utcnow().isoformat(),\n            stage='gguf_cpu_inference',\n            model_name=MODEL_NAME,\n            quantization='Q4_K_M_CPU',\n            vram_mb=0,  # CPU inference uses no GPU VRAM\n            tokens_per_sec_prefill=0,\n            tokens_per_sec_decode=tok_s,\n            perplexity=None,\n            latency_ttft_ms=0,\n            latency_tpot_ms=(total_t*1000)/gen_tokens,\n            notes=f'CPU-only | threads=4 | Load={load_t:.1f}s',\n        ))\nexcept ImportError:\n    print('llama-cpp-python not available. Skipping CPU inference.')\n

## Section 20: Qwen3.5-35B-A3B MoE Optimization (Advanced)\n\nThe 35B-A3B model is a Sparse Mixture-of-Experts (MoE) with 35B total / 3B active parameters.\nAt 4-bit it needs ~17.5GB weights + KV cache, so T4 (16GB) requires aggressive optimization.\n\n### MoE-specific techniques:\n- **Mixed quantization**: attention layers at Q8_0, expert FFNs at IQ4_K\n- **Expert offloading**: route unused experts to CPU RAM\n- **Dynamic batching**: batch multiple tokens per expert call\n- **YaRN scaling**: extend 262K context without recomputing RoPE

In [ ]:
# ============================================================\n# MoE Model Download & Mixed Quantization Recipe\n# ============================================================\nMOE_MODEL = 'Qwen/Qwen3.5-35B-A3B'\nMOE_CACHE = os.path.join(DRIVE_PATH, 'models', MOE_MODEL.replace('/', '_'))\nos.makedirs(MOE_CACHE, exist_ok=True)\n\n# Download only config/tokenizer (skip weights on T4)\nif not os.path.exists(os.path.join(MOE_CACHE, 'config.json')):\n    print(f'Downloading {MOE_MODEL} config/tokenizer...')\n    from huggingface_hub import snapshot_download\n    snapshot_download(\n        repo_id=MOE_MODEL,\n        local_dir=MOE_CACHE,\n        local_dir_use_symlinks=False,\n        allow_patterns=['config.json', 'tokenizer.json', 'tokenizer_config.json', 'preprocessor_config.json'],\n    )\n    print('Config downloaded.')\nelse:\n    print('MoE config already cached.')\n\n# Display MoE architecture info\ntry:\n    from transformers import AutoConfig\n    moe_cfg = AutoConfig.from_pretrained(MOE_CACHE, trust_remote_code=True)\n    print('\\n--- Qwen3.5-35B-A3B Architecture ---')\n    print(f'Hidden size: {moe_cfg.hidden_size}')\n    print(f'Num layers: {moe_cfg.num_hidden_layers}')\n    print(f'Num experts: {getattr(moe_cfg, "num_experts", getattr(moe_cfg, "num_local_experts", "N/A"))}')\n    print(f'Active experts: {getattr(moe_cfg, "num_experts_per_tok", getattr(moe_cfg, "num_active_experts", 8))}')\n    print(f'Vocab size: {moe_cfg.vocab_size}')\n    print(f'Context length: {getattr(moe_cfg, "max_position_embeddings", getattr(moe_cfg, "max_sequence_length", 32768))}')\nexcept Exception as e:\n    print(f'Could not load MoE config: {e}')\n\n# Mixed quantization recipe for llama.cpp GGUF\nprint('\\n--- Recommended GGUF Quant Recipe ---')\nprint('Attention proj:  Q8_0  (preserves accuracy)')\nprint('Expert FFN:      IQ4_K  (aggressive compression)')\nprint('Gate router:     Q6_K   (routing must be precise)')\nprint('KV cache:        Q4_0   (save VRAM during long context)')\nprint('\\nCommand:')\nprint(f'  llama-quantize {MODEL_NAME.replace("\"/\"", "\"_\"")}.gguf output.gguf Q4_K_M')\n

In [ ]:
# ============================================================\n# Expert Offloading & VRAM Simulation\n# ============================================================\ndef simulate_moe_vram(\n    total_params=35_000_000_000,\n    active_params=3_000_000_000,\n    num_experts=256,\n    active_experts=8,\n    bits_per_param=4,\n    kv_cache_tokens=8192,\n    kv_bits=4,\n    hidden_size=3584,\n    num_layers=40,\n):\n    \"\"\"Simulate VRAM usage for MoE with expert offloading.\"\"\"\n    bytes_per_param = bits_per_param / 8\n    # Total weights if all on GPU\n    total_weight_gb = total_params * bytes_per_param / 1e9\n    # Active weights only (offload rest to CPU)\n    active_weight_gb = active_params * bytes_per_param / 1e9\n    # KV cache: 2 (K+V) * num_layers * seq_len * hidden_size * kv_bits/8\n    kv_gb = 2 * num_layers * kv_cache_tokens * hidden_size * (kv_bits / 8) / 1e9\n    # Activations (rough estimate)\n    act_gb = 0.5\n    gpu_total = active_weight_gb + kv_gb + act_gb\n    offloaded_gb = total_weight_gb - active_weight_gb\n    return {\n        'total_weight_gb': total_weight_gb,\n        'active_weight_gb': active_weight_gb,\n        'kv_cache_gb': kv_gb,\n        'activation_gb': act_gb,\n        'gpu_total_gb': gpu_total,\n        'offloaded_gb': offloaded_gb,\n        'offload_ratio': offloaded_gb / total_weight_gb,\n        'fits_t4': gpu_total < 16,\n        'fits_a100': gpu_total < 40,\n    }\n\nscenarios = [\n    {'name': '35B-A3B @ 4-bit (all experts)', 'bits': 4, 'offload': False},\n    {'name': '35B-A3B @ 4-bit (expert offloading)', 'bits': 4, 'offload': True},\n    {'name': '35B-A3B @ 3-bit (expert offloading)', 'bits': 3, 'offload': True},\n]\n\nprint('\\n' + '='*70)\nprint('MoE VRAM Simulation for Qwen3.5-35B-A3B')\nprint('='*70)\nfor s in scenarios:\n    sim = simulate_moe_vram(bits_per_param=s['bits'])\n    if s['offload']:\n        gpu = sim['active_weight_gb'] + sim['kv_cache_gb'] + sim['activation_gb']\n    else:\n        gpu = sim['gpu_total_gb']\n    print(f'\\n{s["name"]}:')\n    print(f'  GPU VRAM:  {gpu:.2f} GB')\n    print(f'  CPU RAM:   {sim["offloaded_gb"]:.2f} GB (offloaded)')\n    print(f'  KV cache:  {sim["kv_cache_gb"]:.2f} GB')\n    print(f'  Fits T4:   {gpu < 16}')\n    print(f'  Fits A100: {gpu < 40}')\nprint('='*70)\n

In [ ]:
# ============================================================\n# MoE Expert Routing Visualization\n# ============================================================\nimport matplotlib.pyplot as plt\nimport numpy as np\n\ndef visualize_expert_routing(num_experts=256, active_experts=8, seq_len=64):\n    \"\"\"Visualize which experts are active across sequence positions.\"\"\"\n    # Simulate random routing (in real model this comes from gate network)\n    np.random.seed(42)\n    routing = np.zeros((seq_len, num_experts))\n    for i in range(seq_len):\n        active = np.random.choice(num_experts, active_experts, replace=False)\n        routing[i, active] = 1\n\n    plt.figure(figsize=(12, 4))\n    plt.imshow(routing, aspect='auto', cmap='Blues', interpolation='nearest')\n    plt.xlabel('Expert Index')\n    plt.ylabel('Sequence Position')\n    plt.title(f'MoE Expert Routing ({num_experts} experts, {active_experts} active/token)')\n    plt.colorbar(label='Active')\n    plt.tight_layout()\n    plt.savefig(os.path.join(DRIVE_PATH, 'moe_routing.png'), dpi=150)\n    plt.show()\n    print(f'Saved routing visualization to {os.path.join(DRIVE_PATH, "moe_routing.png")}')\n\nvisualize_expert_routing()\n\n# Compute expert utilization statistics\ndef expert_utilization_stats(num_experts=256, active_experts=8, tokens=1000):\n    counts = np.zeros(num_experts)\n    for _ in range(tokens):\n        active = np.random.choice(num_experts, active_experts, replace=False)\n        counts[active] += 1\n    utilization = counts / tokens\n    print('\\n--- Expert Utilization Stats ---')\n    print(f'Mean utilization: {utilization.mean():.4f}')\n    print(f'Std utilization:  {utilization.std():.4f}')\n    print(f'Max utilization:  {utilization.max():.4f}')\n    print(f'Min utilization:  {utilization.min():.4f}')\n    print(f'Unused experts:   {(counts == 0).sum()} / {num_experts}')\n    # Experts that could be pruned (never used)\n    pruneable = (counts == 0).sum()\n    print(f'Pruneable experts: {pruneable} ({pruneable/num_experts*100:.1f}%)')\n\nexpert_utilization_stats()\n

In [ ]:
# ============================================================
# Save notebook copy to Drive
# ============================================================
import os
NOTEBOOK_BACKUP = os.path.join(DRIVE_PATH, "colab_qwen_optimize_backup.ipynb")
print(f"Notebook backup path: {NOTEBOOK_BACKUP}")
print("Use File → Save a copy in Drive to keep the interactive version.")

logger.summary()
print("
Done!")
